In [1]:
import pandas as pd

ml_df = pd.read_csv(
    "../data/processed/jobs_cleaned.csv"
)

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/jobs_cleaned.csv'

In [2]:
import os

print(os.getcwd())

c:\Users\Aarya Pandhare\OneDrive\Desktop\ai-job-salary-intelligence\notebooks


In [3]:
import pandas as pd

ml_df = pd.read_csv("../data/processed/jobs_cleaned.csv")

print(ml_df.shape)
print(ml_df.head())

(395, 16)
       job_id        search_role  \
0  5912847538  software engineer   
1  5917523641  software engineer   
2  5914221304  software engineer   
3  5914221345  software engineer   
4  5832769660  software engineer   

                                             title  \
0  Software Engineering - Custom Software Engineer   
1  Software Engineering - Custom Software Engineer   
2  Software Engineering - Custom Software Engineer   
3  Software Engineering - Custom Software Engineer   
4                                Software Engineer   

                            company                  location category  \
0  Trigent Software Private Limited   Hussainialam, Hyderabad  IT Jobs   
1  Trigent Software Private Limited  Richmond Town, Bangalore  IT Jobs   
2  Trigent Software Private Limited  Richmond Town, Bangalore  IT Jobs   
3  Trigent Software Private Limited  Richmond Town, Bangalore  IT Jobs   
4                          Capawoke      Bangalore, Karnataka  IT Jobs   

  c

In [4]:
from sklearn.model_selection import train_test_split

X = ml_df[["title", "description", "location"]].fillna("")
y = ml_df["salary_midpoint"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 316
Testing records: 79


In [5]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

baseline_model = DummyRegressor(strategy="median")

baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)

print("Baseline MAE (INR):", round(baseline_mae, 2))
print("Baseline MAE (lakhs):", round(baseline_mae / 100000, 2))

Baseline MAE (INR): 756075.9
Baseline MAE (lakhs): 7.56


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

job_titles = [
    "Python Developer",
    "Python Engineer",
    "Java Developer"
]

vectorizer = TfidfVectorizer()

X_example = vectorizer.fit_transform(job_titles)

print("Features:")
print(vectorizer.get_feature_names_out())

print("\nTF-IDF Matrix:")
print(X_example.toarray().round(2))

Features:
['developer' 'engineer' 'java' 'python']

TF-IDF Matrix:
[[0.71 0.   0.   0.71]
 [0.   0.8  0.   0.61]
 [0.61 0.   0.8  0.  ]]


In [7]:
import pandas as pd

tfidf_df = pd.DataFrame(
    X_example.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=job_titles
)

print(tfidf_df.round(2))

                  developer  engineer  java  python
Python Developer       0.71       0.0   0.0    0.71
Python Engineer        0.00       0.8   0.0    0.61
Java Developer         0.61       0.0   0.8    0.00


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

title_model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("ridge", Ridge(alpha=1.0))
])

title_model.fit(X_train["title"], y_train)

title_predictions = title_model.predict(X_test["title"])

title_mae = mean_absolute_error(
    y_test,
    title_predictions
)

print("Baseline MAE:", round(baseline_mae / 100000, 2), "lakhs")
print("Title Model MAE:", round(title_mae / 100000, 2), "lakhs")

Baseline MAE: 7.56 lakhs
Title Model MAE: 7.13 lakhs


In [9]:
X_train_text = (
    X_train["title"]
    + " "
    + X_train["description"]
)

X_test_text = (
    X_test["title"]
    + " "
    + X_test["description"]
)

print(X_train_text.iloc[0][:500])

Data Scientist Position: Data Scientist Location: Trivandrum (on-site) Experience Required: 5 Years Budget: Upto 30 LPA About the Role The role requires a strong focus on data analysis, machine learning model development, and fraud detection across large-scale datasets. The ideal candidate collaborates closely with engineering and product teams to build scalable and reliable machine learning solutions that support data-driven decision-making. Exposure to model development, feature engineering, e


In [10]:
text_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=3000,
        stop_words="english"
    )),
    ("ridge", Ridge(alpha=1.0))
])

text_model.fit(
    X_train_text,
    y_train
)

text_predictions = text_model.predict(
    X_test_text
)

text_mae = mean_absolute_error(
    y_test,
    text_predictions
)

print("Baseline MAE:", round(baseline_mae / 100000, 2), "lakhs")
print("Title Model MAE:", round(title_mae / 100000, 2), "lakhs")
print("Title + Description MAE:", round(text_mae / 100000, 2), "lakhs")

Baseline MAE: 7.56 lakhs
Title Model MAE: 7.13 lakhs
Title + Description MAE: 7.25 lakhs


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

In [12]:
preprocessor = ColumnTransformer([
    (
        "title_tfidf",
        TfidfVectorizer(),
        "title"
    ),
    (
        "location_encoder",
        OneHotEncoder(handle_unknown="ignore"),
        ["location"]
    )
])

In [13]:
location_model = Pipeline([
    ("preprocessor", preprocessor),
    ("ridge", Ridge(alpha=1.0))
])

location_model.fit(X_train, y_train)

location_predictions = location_model.predict(X_test)

location_mae = mean_absolute_error(
    y_test,
    location_predictions
)

print("Baseline MAE:", round(baseline_mae / 100000, 2))
print("Title MAE:", round(title_mae / 100000, 2))
print("Title + Description MAE:", round(text_mae / 100000, 2))
print("Title + Location MAE:", round(location_mae / 100000, 2))

Baseline MAE: 7.56
Title MAE: 7.13
Title + Description MAE: 7.25
Title + Location MAE: 6.93


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

full_preprocessor = ColumnTransformer([
    (
        "title_tfidf",
        TfidfVectorizer(),
        "title"
    ),
    (
        "description_tfidf",
        TfidfVectorizer(
            max_features=3000,
            stop_words="english"
        ),
        "description"
    ),
    (
        "location_encoder",
        OneHotEncoder(handle_unknown="ignore"),
        ["location"]
    )
])

In [15]:
full_model = Pipeline([
    ("preprocessor", full_preprocessor),
    ("ridge", Ridge(alpha=1.0))
])

full_model.fit(X_train, y_train)

full_predictions = full_model.predict(X_test)

full_mae = mean_absolute_error(
    y_test,
    full_predictions
)

print("Baseline MAE:", round(baseline_mae / 100000, 2))
print("Title MAE:", round(title_mae / 100000, 2))
print("Title + Description MAE:", round(text_mae / 100000, 2))
print("Title + Location MAE:", round(location_mae / 100000, 2))
print("All Features MAE:", round(full_mae / 100000, 2))

Baseline MAE: 7.56
Title MAE: 7.13
Title + Description MAE: 7.25
Title + Location MAE: 6.93
All Features MAE: 7.03


In [16]:
train_predictions = full_model.predict(X_train)

train_mae = mean_absolute_error(
    y_train,
    train_predictions
)

print("Training MAE:", round(train_mae / 100000, 2), "lakhs")
print("Testing MAE:", round(full_mae / 100000, 2), "lakhs")

Training MAE: 3.24 lakhs
Testing MAE: 7.03 lakhs


In [17]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    location_model,
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error"
)

cv_mae = -cv_scores / 100000

print("MAE for each fold:", cv_mae.round(2))
print("Average CV MAE:", round(cv_mae.mean(), 2), "lakhs")
print("Standard Deviation:", round(cv_mae.std(), 2), "lakhs")

MAE for each fold: [6.59 6.46 7.98 6.88 6.41]
Average CV MAE: 6.86 lakhs
Standard Deviation: 0.58 lakhs


In [18]:
from sklearn.model_selection import cross_val_score
import numpy as np

# Model 1: Title + Location
location_cv = -cross_val_score(
    location_model,
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error"
) / 100000

# Model 2: Title + Description + Location
full_cv = -cross_val_score(
    full_model,
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error"
) / 100000

print("Title + Location CV:", location_cv.round(2))
print("All Features CV:", full_cv.round(2))

print("\nAverage CV MAE:")
print("Title + Location:", round(location_cv.mean(), 2), "lakhs")
print("All Features:", round(full_cv.mean(), 2), "lakhs")

print("\nStandard Deviation:")
print("Title + Location:", round(location_cv.std(), 2))
print("All Features:", round(full_cv.std(), 2))

Title + Location CV: [6.59 6.46 7.98 6.88 6.41]
All Features CV: [6.41 5.88 7.49 6.99 6.08]

Average CV MAE:
Title + Location: 6.86 lakhs
All Features: 6.57 lakhs

Standard Deviation:
Title + Location: 0.58
All Features: 0.6
